# RAG Pipeline Demo (Simple Version)

Beginner-friendly, straight-line notebook. No function definitions, no try/except.

Steps: Load document -> Split into chunks -> Create embeddings -> Store in FAISS -> Search.

In [1]:
# Step 0: Imports
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

C:\Users\sriam\AppData\Roaming\Python\Python314\site-packages\langchain_core\utils\pydantic.py:41: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


ModuleNotFoundError: No module named 'langchain_huggingface'

## Step 1: Load the document

We read `sample_document.txt` into a LangChain `Document` object.

In [ ]:
loader = TextLoader("sample_document.txt", encoding="utf-8")
documents = loader.load()

print("Loaded", len(documents), "document(s)")
print(documents[0].page_content[:300])

## Step 2: Split the document into chunks

Big documents are split into smaller overlapping chunks so the embedding model can handle them.

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)
chunks = splitter.split_documents(documents)

print("Created", len(chunks), "chunks")
print(chunks[0].page_content)

## Step 3: Create embeddings

An embedding model turns text into a list of numbers (a vector) that captures its meaning.

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

sample_vector = embeddings.embed_query("What is RAG?")
print("Vector length:", len(sample_vector))
print(sample_vector[:5])

## Step 4: Store chunk vectors in FAISS

FAISS is a local vector database. It stores all chunk vectors so we can search them later.

In [ ]:
vector_db = FAISS.from_documents(chunks, embeddings)

print("FAISS index created with", vector_db.index.ntotal, "vectors")

## Step 5: Search for similar chunks

We turn a question into a vector and find the closest chunk vectors. A lower score means a closer (more similar) match.

In [ ]:
query = "What is Retrieval-Augmented Generation?"
results = vector_db.similarity_search_with_score(query, k=3)

for doc, score in results:
    print("Score:", score)
    print(doc.page_content)
    print("-" * 40)

## Summary

In this notebook:
1. Loaded a text document.
2. Split it into overlapping chunks.
3. Created embeddings using a Hugging Face model.
4. Stored the chunk vectors in a FAISS index.
5. Searched the index with a natural-language query.

All code here is straight-line: no function definitions, no try/except blocks.